# Train "CustomModel 5.1" — SpecDecode draft model

Fine-tunes **Llama-3.2-1B-Instruct** into a custom draft model for the SpecDecode speculative-decoding app, then exports it straight to a quantized `.gguf` you download and drop into the project's `models/` folder — it shows up in both dropdowns automatically, no other setup needed.

**Before you start:** In Colab, go to `Runtime > Change runtime type` and pick a **T4 GPU** (or better). This notebook trains in bf16 LoRA via [Unsloth](https://github.com/unslothai/unsloth), which is ~2x faster and much lighter on VRAM than plain HF training — a full T4 (16GB) comfortably fits a 1B model with plenty of headroom for a real batch size.

**What this does, end to end:**
1. Loads Llama-3.2-1B-Instruct + adds a LoRA adapter
2. Loads a public instruction dataset (Alpaca, ~50k examples) and formats it with the *exact same* Llama-3 chat template the app uses at inference time (`server/llamaClient.ts`'s `formatLlama3Prompt`) — matching train/serve format matters for a good acceptance rate
3. Trains, saves the LoRA adapter (so you can continue training later with a second dataset — "round 2")
4. Merges + quantizes to `CustomModel 5.1.gguf` (Q4_K_M, matching the quantization of the models already in this project)
5. Downloads it to your machine

## 1. Install dependencies

In [ ]:
%%capture
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — go to Runtime > Change runtime type and pick a GPU")
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else "-")

## 2. Config — edit these if you want, defaults match the plan discussed

In [ ]:
BASE_MODEL = "unsloth/Llama-3.2-1B-Instruct"   # ungated mirror of the same weights already running locally as GGUF
OUTPUT_NAME = "CustomModel 5.1"                # exact name shown in the SpecDecode app's dropdown

# Round 1 dataset
DATASET_NAME = "tatsu-lab/alpaca"
MAX_EXAMPLES = 50_000
NUM_EPOCHS = 1

MAX_SEQ_LENGTH = 1024      # matches typical Alpaca example length; app's default ctx is 2048
LORA_RANK = 16
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4       # effective batch size = 16

ADAPTER_DIR = "/content/customModel_5_1_adapter"   # save this to Drive (cell below) if you want to do round 2 later

## 3. Load base model + attach LoRA adapter

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-detect (bf16 on T4/A100)
    load_in_4bit=False,  # plenty of VRAM on a T4 for a 1B model in bf16 — keeps full quality
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 4. Load + format the dataset

Formats every example through the **same Llama-3 chat template** `server/llamaClient.ts` uses when serving real requests. This is the detail that actually matters for acceptance rate: the draft model needs to see prompts shaped exactly like what it'll be asked at inference time.

In [ ]:
from datasets import load_dataset

SYSTEM_PROMPT = "You are a helpful, concise assistant."

def format_example(example):
    instruction = example["instruction"]
    input_text = example.get("input", "") or ""
    user_content = f"{instruction}\n\n{input_text}" if input_text else instruction
    text = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"{SYSTEM_PROMPT}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_content}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{example['output']}<|eot_id|>"
    )
    return {"text": text}

dataset = load_dataset(DATASET_NAME, split="train")
if MAX_EXAMPLES and len(dataset) > MAX_EXAMPLES:
    dataset = dataset.shuffle(seed=3407).select(range(MAX_EXAMPLES))
dataset = dataset.map(format_example, remove_columns=dataset.column_names)

print(f"{len(dataset)} training examples")
print("--- Sample formatted example ---")
print(dataset[0]["text"])

## 5. (Optional) Smoke test — train on 200 examples first

Uncomment and run this before the full run to catch any config/environment issue in a couple minutes instead of after a long run.

In [ ]:
# SMOKE_TEST = True
SMOKE_TEST = False

train_dataset = dataset.select(range(200)) if SMOKE_TEST else dataset

## 6. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=1 if SMOKE_TEST else NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_steps=200,
        save_total_limit=2,
        output_dir="/content/train_checkpoints",
        optim="adamw_8bit",
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        report_to="none",
    ),
)

train_result = trainer.train()
print(train_result)

## 7. Save the LoRA adapter

This is what you'd reload for **round 2** (continuing training with a second ~50k dataset later) — save it to Google Drive so it survives after this Colab session ends.

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

In [ ]:
# Optional but recommended: persist the adapter to Google Drive so round 2
# can resume from it even in a fresh Colab session.
SAVE_ADAPTER_TO_DRIVE = True

if SAVE_ADAPTER_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    drive_dest = "/content/drive/MyDrive/SpecDecode/customModel_5_1_adapter"
    shutil.copytree(ADAPTER_DIR, drive_dest, dirs_exist_ok=True)
    print(f"Copied adapter to {drive_dest}")

## 8. Merge + export to GGUF (Q4_K_M)

Matches the quantization of the models already in the SpecDecode project, so behavior/VRAM usage is consistent with `Llama-3.2-3B-Instruct-Q4_K_M.gguf` / `Llama-3.2-1B-Instruct-Q4_K_M.gguf`.

In [ ]:
import glob, os, shutil

GGUF_EXPORT_DIR = "/content/gguf_export"

model.save_pretrained_gguf(GGUF_EXPORT_DIR, tokenizer, quantization_method="q4_k_m")

# unsloth's exact output filename varies by version, so find whatever .gguf it produced
produced = glob.glob(f"{GGUF_EXPORT_DIR}*.gguf") + glob.glob(f"{GGUF_EXPORT_DIR}/*.gguf") + glob.glob("/content/*q4_k_m*.gguf")
if not produced:
    raise FileNotFoundError("No .gguf found — check the save_pretrained_gguf output above for the actual path.")

final_path = f"/content/{OUTPUT_NAME}.gguf"
shutil.copy(produced[0], final_path)
print(f"Final model: {final_path} ({os.path.getsize(final_path) / 1e9:.2f} GB)")

## 9. Download

After this downloads, copy the file into your project's `models/` folder:
```bash
# on your machine
mv "~/Downloads/CustomModel 5.1.gguf" "C:/Users/noah6/Downloads/SpecDecode-main/SpecDecode-main/models/"
```
It'll appear automatically in both dropdowns in the Live Demo — no restart needed.

In [ ]:
from google.colab import files
files.download(final_path)

---
## Round 2 (later): continue training with a second ~50k dataset

Run this in a **new** Colab session, after re-running the install cell (section 1) and the config/base-model-load cells (2–3), but instead of loading a fresh LoRA adapter, load the one you saved to Drive and keep training it — then re-export, which **overwrites** `CustomModel 5.1.gguf` with the further-trained version (per your earlier choice, no separate v2 file).

In [ ]:
# --- ROUND 2: uncomment and run after round 1 is done and you have a second dataset ready ---

# from google.colab import drive
# drive.mount("/content/drive")
#
# from unsloth import FastLanguageModel
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name="/content/drive/MyDrive/SpecDecode/customModel_5_1_adapter",  # resumes from your round-1 adapter
#     max_seq_length=MAX_SEQ_LENGTH,
#     dtype=None,
#     load_in_4bit=False,
# )
#
# DATASET_NAME_ROUND2 = "yahma/alpaca-cleaned"   # distinct from round 1's tatsu-lab/alpaca
# dataset_r2 = load_dataset(DATASET_NAME_ROUND2, split="train")
# dataset_r2 = dataset_r2.shuffle(seed=3407).select(range(50_000))
# dataset_r2 = dataset_r2.map(format_example, remove_columns=dataset_r2.column_names)
#
# trainer_r2 = SFTTrainer(
#     model=model, tokenizer=tokenizer, train_dataset=dataset_r2,
#     dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH, packing=False,
#     args=TrainingArguments(
#         per_device_train_batch_size=PER_DEVICE_BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM_STEPS,
#         num_train_epochs=1, learning_rate=LEARNING_RATE,
#         bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
#         logging_steps=10, output_dir="/content/train_checkpoints_r2", optim="adamw_8bit",
#         lr_scheduler_type="cosine", warmup_ratio=0.03, report_to="none",
#     ),
# )
# trainer_r2.train()
#
# # Re-save adapter to Drive (overwrite), then re-run section 8's export cell to overwrite the .gguf
# model.save_pretrained("/content/customModel_5_1_adapter_r2")
# tokenizer.save_pretrained("/content/customModel_5_1_adapter_r2")
# import shutil
# shutil.copytree("/content/customModel_5_1_adapter_r2", "/content/drive/MyDrive/SpecDecode/customModel_5_1_adapter", dirs_exist_ok=True)